## 1. load datasets

### load first Discharge dataset

In [3]:
import pandas as pd

path = r'D:\clinical text dataset\discharge.csv'

df = pd.read_csv(path, nrows=1000)

print(df.shape)
print(df.columns)
print(df.head())

(1000, 8)
Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text'],
      dtype='object')
          note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   
1  10000032-DS-22    10000032  22841357        DS        22   
2  10000032-DS-23    10000032  29079034        DS        23   
3  10000032-DS-24    10000032  25742920        DS        24   
4  10000084-DS-17    10000084  23052089        DS        17   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   
1  2180-06-27 00:00:00  2180-07-01 10:15:00   
2  2180-07-25 00:00:00  2180-07-25 21:42:00   
3  2180-08-07 00:00:00  2180-08-10 05:43:00   
4  2160-11-25 00:00:00  2160-11-25 15:09:00   

                                                text  
0   \nName:  ___                     Unit No:   _...  
1   \nName:  ___                     Unit No:   _...  
2   \nName:  ___               

### Radiology dataset load

In [5]:
import pandas as pd

df2 = pd.read_csv(r'D:\clinical text dataset\radiology.csv', nrows=5)

print(df2.shape)
print(df2.columns)
print(df2.head())

(5, 8)
Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text'],
      dtype='object')
          note_id  subject_id     hadm_id note_type  note_seq  \
0  10000032-RR-14    10000032  22595853.0        RR        14   
1  10000032-RR-15    10000032  22595853.0        RR        15   
2  10000032-RR-16    10000032  22595853.0        RR        16   
3  10000032-RR-18    10000032         NaN        RR        18   
4  10000032-RR-20    10000032         NaN        RR        20   

             charttime            storetime  \
0  2180-05-06 21:19:00  2180-05-06 23:32:00   
1  2180-05-06 23:00:00  2180-05-06 23:26:00   
2  2180-05-07 09:55:00  2180-05-07 11:15:00   
3  2180-06-03 12:46:00  2180-06-03 14:01:00   
4  2180-07-08 13:18:00  2180-07-08 14:15:00   

                                                text  
0  EXAMINATION:  CHEST (PA AND LAT)\n\nINDICATION...  
1  EXAMINATION:  LIVER OR GALLBLADDER US (SINGLE ...  
2  INDICATION:  ___ HC

## 2. EDA

In [8]:
import pandas as pd

# ── Load both
discharge = pd.read_csv(r'D:\clinical text dataset\discharge.csv')
radiology = pd.read_csv(r'D:\clinical text dataset\radiology.csv')

print("=" * 50)
print("DISCHARGE NOTES")
print("=" * 50)
print(f"Shape          : {discharge.shape}")
print(f"Unique patients: {discharge['subject_id'].nunique():,}")
print(f"Unique admissions: {discharge['hadm_id'].nunique():,}")
print(f"Null in text   : {discharge['text'].isnull().sum()}")

print("\n" + "=" * 50)
print("RADIOLOGY NOTES")
print("=" * 50)
print(f"Shape          : {radiology.shape}")
print(f"Unique patients: {radiology['subject_id'].nunique():,}")
print(f"Null in text   : {radiology['text'].isnull().sum()}")

# ── Sample text dekhte hain
print("\n── Sample discharge note (first 500 chars) ──")
print(discharge['text'].iloc[0][:500])

print("\n── Sample radiology note (first 500 chars) ──")
print(radiology['text'].iloc[0][:500])

DISCHARGE NOTES
Shape          : (331793, 8)
Unique patients: 145,914
Unique admissions: 331,793
Null in text   : 0

RADIOLOGY NOTES
Shape          : (2321355, 8)
Unique patients: 237,427
Null in text   : 0

── Sample discharge note (first 500 chars) ──
 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergies / Adverse Drug Reactions
 
Attending: ___
 
Chief Complaint:
Worsening ABD distension and pain 
 
Major Surgical or Invasive Procedure:
Paracentesis

 
History of Present Illness:
___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED with worsening abd 
d

── Sample radiology note (first 500 chars) ──
EXAMINATION:  CHEST (PA AND LAT)

INDICATION:  ___ with new onset ascites  // eval for infection

TECHNIQUE:  Chest PA and lateral

COMPARISON:  None.

FINDINGS: 

There is no focal consolidation, p

In [9]:
# ── Text length analysis
discharge['text_len'] = discharge['text'].str.len()
radiology['text_len'] = radiology['text'].str.len()

print("Discharge text length stats:")
print(discharge['text_len'].describe().round(0))

print("\nRadiology text length stats:")
print(radiology['text_len'].describe().round(0))

Discharge text length stats:
count    331793.0
mean      10551.0
std        4452.0
min         353.0
25%        7462.0
50%        9847.0
75%       12831.0
max       60381.0
Name: text_len, dtype: float64

Radiology text length stats:
count    2321355.0
mean        1159.0
std         1005.0
min            3.0
25%          511.0
50%          781.0
75%         1440.0
max        35849.0
Name: text_len, dtype: float64


## 3. cardiac filter

In [1]:
import pandas as pd
import gc

path = r'D:\clinical text dataset\discharge.csv'

# ── first only structure check
df_sample = pd.read_csv(
    path,
    nrows    = 100,
    engine   = 'c',           # default C parser
    encoding = 'utf-8'
)
print("Columns:", df_sample.columns.tolist())
print("Shape sample:", df_sample.shape)
print("\nFirst text (200 chars):")
print(repr(df_sample['text'].iloc[0][:200]))

Columns: ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime', 'text']
Shape sample: (100, 8)

First text (200 chars):
' \nName:  ___                     Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nNo Known Allergie'


In [4]:
import pandas as pd
import re
import gc

# ── Specific cardiac/arrhythmia keywords — tighter list
cardiac_keywords = [
    'arrhythmia', 'atrial fibrillation', 'atrial flutter',
    'afib', 'a-fib',
    'ventricular fibrillation', 'ventricular tachycardia',
    'supraventricular tachycardia', 'svt',
    'tachycardia', 'bradycardia',
    'heart block', 'bundle branch block',
    'premature ventricular', 'premature atrial',
    'palpitation', 'electrocardiogram', 'ecg showed',
    'ekg showed', 'sinus rhythm', 'sinus tachycardia',
    'sinus bradycardia', 'qrs', 'qt prolongation',
    'wolff-parkinson-white', 'wpw',
    'cardiomyopathy', 'myocardial infarction'
]

# \b word boundary — exact match only
pattern = r'\b(?:' + '|'.join(
    re.escape(kw) for kw in cardiac_keywords
) + r')\b'

path = r'D:\clinical text dataset\discharge.csv'

cardiac_rows  = []
total_notes   = 0
cardiac_count = 0

print("Processing with strict word-boundary matching...")

for i, chunk in enumerate(pd.read_csv(
        path,
        chunksize    = 5000,
        engine       = 'c',
        encoding     = 'utf-8',
        on_bad_lines = 'skip',
        dtype        = str
)):
    if 'text' not in chunk.columns:
        continue

    total_notes  += len(chunk)
    text_lower    = chunk['text'].fillna('').str.lower()
    mask          = text_lower.str.contains(
                        pattern, na=False, regex=True
                    )
    cardiac_count += mask.sum()

    cardiac_rows.append(
        chunk[mask][['note_id', 'subject_id',
                     'hadm_id', 'text']].copy()
    )

    del text_lower, mask
    gc.collect()

    if (i + 1) % 10 == 0:
        print(f"  {total_notes:,} notes | cardiac: {cardiac_count:,}")

print(f"\nTotal notes : {total_notes:,}")
print(f"Cardiac     : {cardiac_count:,}  "
      f"({cardiac_count/total_notes*100:.1f}%)")

cardiac_df = pd.concat(cardiac_rows, ignore_index=True)
del cardiac_rows
gc.collect()

print(f"Unique patients: {cardiac_df['subject_id'].nunique():,}")

# ── Per keyword sample check
print("\n── Per keyword count (10k sample) ──")
sample = cardiac_df['text'].sample(
    n=min(10000, len(cardiac_df)), random_state=42
).str.lower()

for kw in cardiac_keywords:
    pat = r'\b' + re.escape(kw) + r'\b'
    cnt = sample.str.contains(pat, na=False, regex=True).sum()
    print(f"  {kw:<35}: {cnt:,}")

del sample
gc.collect()

# ── Save — overwrite old file
cardiac_df.to_csv('cardiac_discharge.csv', index=False)
print(f"\nSaved: cardiac_discharge.csv ({len(cardiac_df):,} rows)")

Processing with strict word-boundary matching...
  50,000 notes | cardiac: 19,822
  100,000 notes | cardiac: 39,731
  150,000 notes | cardiac: 59,635
  200,000 notes | cardiac: 79,570
  250,000 notes | cardiac: 99,274
  300,000 notes | cardiac: 119,009

Total notes : 331,793
Cardiac     : 131,390  (39.6%)
Unique patients: 67,351

── Per keyword count (10k sample) ──
  arrhythmia                         : 1,274
  atrial fibrillation                : 3,553
  atrial flutter                     : 380
  afib                               : 2,672
  a-fib                              : 361
  ventricular fibrillation           : 36
  ventricular tachycardia            : 226
  supraventricular tachycardia       : 128
  svt                                : 350
  tachycardia                        : 2,824
  bradycardia                        : 1,228
  heart block                        : 331
  bundle branch block                : 181
  premature ventricular              : 37
  premature atrial   

## 4. Text Clining

In [5]:
import pandas as pd
import re
import gc

cardiac_df = pd.read_csv('cardiac_discharge.csv', dtype=str)
print(f"Loaded: {len(cardiac_df):,} rows")

def clean_clinical_text(text):
    if not isinstance(text, str):
        return ""
    
    # deidentification placeholders
    text = re.sub(r'___+', ' ', text)
    
    # improved header removal (more robust)
    text = re.sub(
        r'(name|unit no|admission date|discharge date|date of birth|sex|service|attending)[\s:]+[^\n]*',
        '', text, flags=re.IGNORECASE
    )
    
    # newlines → space
    text = re.sub(r'\n+', ' ', text)
    
    # extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    text = text.lower().strip()
    return text

print("Cleaning...")

# safer apply (handles NaN)
cardiac_df['text_clean'] = cardiac_df['text'].astype(str).apply(clean_clinical_text)

cardiac_df['clean_len']  = cardiac_df['text_clean'].str.len().astype(int)

# Short notes remove
before = len(cardiac_df)
cardiac_df = cardiac_df[cardiac_df['clean_len'] >= 200]

print(f"Removed short : {before - len(cardiac_df):,}")
print(f"Remaining     : {len(cardiac_df):,}")

# % drop check
print(f"Drop %        : {(before - len(cardiac_df)) / before * 100:.2f}%")

# remove duplicates (IMPORTANT)
cardiac_df = cardiac_df.drop_duplicates(subset=['text_clean'])

# reset index
cardiac_df = cardiac_df.reset_index(drop=True)

# sanity check
print("\nNull check:")
print(cardiac_df.isnull().sum())

print("\nCleaned length stats:")
print(cardiac_df['clean_len'].describe().round(0))

print("\n── Sample cleaned note (600 chars) ──")
print(cardiac_df['text_clean'].iloc[0][:600])

# Save
cardiac_df[['note_id','subject_id',
            'hadm_id','text_clean']].to_csv(
    'cardiac_discharge_clean.csv', index=False
)

print(f"\nSaved: cardiac_discharge_clean.csv ({len(cardiac_df):,} rows)")

Loaded: 131,390 rows
Cleaning...
Removed short : 0
Remaining     : 131,390
Drop %        : 0.00%

Null check:
note_id       0
subject_id    0
hadm_id       0
text          0
text_clean    0
clean_len     0
dtype: int64

Cleaned length stats:
count    131390.0
mean      11329.0
std        4342.0
min         629.0
25%        8352.0
50%       10648.0
75%       13489.0
max       55429.0
Name: clean_len, dtype: float64

── Sample cleaned note (600 chars) ──
allergies: percocet / vicodin altered mental status major surgical or invasive procedure: none history of present illness: mrs. is a female with hiv on haart, copd, hcv cirrhosis complicated by ascites and hepatic encephalopathy who initially presented to the ed yesterday with hypotension after a paracentesis. the patient has had accelerated decompensation of her cirrhosis recently with worsening ascites, and she is maintained on twice weekly paracentesis. she was at her regular session yesterday when she had hypotension to sbp and felt 

In [6]:
! pip install transformers sentencepiece

   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   --------- ------------------------------ 2.4/10.5 MB 14.9 MB/s eta 0:00:01
   ---------------------------------------  10.2/10.5 MB 30.4 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 19.2 MB/s  0:00:00
   ---------------------------------------- 0.0/660.6 kB ? eta -:--:--
   ---------------------------------------- 660.6/660.6 kB 4.3 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------  3.7/3.7 MB 54.1 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 12.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 17.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 6.3 MB/s  0:00:00

   ----------------------------------------

## 5. ClinicalBert + Sliding Window

In [7]:
import pandas as pd
from transformers import AutoTokenizer
import numpy as np

# ClinicalBERT —  pretrained for medical text
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test on one note
sample_text = cardiac_df['text_clean'].iloc[0]
tokens = tokenizer(sample_text, truncation=False)
print(f"Sample note tokens : {len(tokens['input_ids'])}")
print(f"BERT max           : 512")
print(f"Chunks needed      : {len(tokens['input_ids']) // 512 + 1}")

d:\ecg_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\ecg_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\AI_Models\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.micros

Sample note tokens : 3009
BERT max           : 512
Chunks needed      : 6


In [8]:
# Token length distribution check — how many chucks required
cardiac_df2 = pd.read_csv('cardiac_discharge_clean.csv', dtype=str)

sample_100 = cardiac_df2['text_clean'].sample(100, random_state=42)

token_lengths = []
for text in sample_100:
    toks = tokenizer(text, truncation=False)
    token_lengths.append(len(toks['input_ids']))

token_lengths = np.array(token_lengths)
print(f"Token length stats (100 sample):")
print(f"  Mean   : {token_lengths.mean():.0f}")
print(f"  Max    : {token_lengths.max():.0f}")
print(f"  Median : {np.median(token_lengths):.0f}")
print(f"  >512   : {(token_lengths > 512).sum()} / 100")
print(f"  >1024  : {(token_lengths > 1024).sum()} / 100")

Token length stats (100 sample):
  Mean   : 3343
  Max    : 8479
  Median : 3159
  >512   : 100 / 100
  >1024  : 98 / 100


## Clinical Text Embedding using BERT

In [10]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import gc
import os

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
bert_model.eval()
print(f"Device: {device}")

# ────────────────────────────────────────────────────────────────
# Batch-aware sliding window embedding
# ────────────────────────────────────────────────────────────────
def tokenize_with_chunks(texts, max_len=512, stride=256, max_chunks=6):
    """
    List of texts → each text ke chunks tokenize karo
    Returns: list of list of (input_ids, attention_mask)
    """
    all_chunks = []
    for text in texts:
        tokens = tokenizer(
            text,
            return_tensors = 'pt',
            truncation     = False,
            padding        = False
        )
        ids  = tokens['input_ids'][0]
        mask = tokens['attention_mask'][0]
        total = len(ids)

        text_chunks = []
        start = 0
        count = 0
        while start < total and count < max_chunks:
            end = min(start + max_len, total)
            text_chunks.append((
                ids[start:end],
                mask[start:end]
            ))
            start += stride
            count += 1

        all_chunks.append(text_chunks)
    return all_chunks


def embed_batch(texts, max_len=512, stride=256, max_chunks=6):
    """
    Batch of texts → embeddings (batch_size, 768)
    Har text ke chunks → mean pooling → final embedding
    """
    all_chunks = tokenize_with_chunks(
        texts, max_len, stride, max_chunks
    )
    batch_embeddings = []

    for text_chunks in all_chunks:
        chunk_embs = []

        for ids, mask in text_chunks:
            # pad to max_len
            pad_len = max_len - len(ids)
            if pad_len > 0:
                ids  = torch.cat([
                    ids,
                    torch.zeros(pad_len, dtype=torch.long)
                ])
                mask = torch.cat([
                    mask,
                    torch.zeros(pad_len, dtype=torch.long)
                ])

            ids  = ids.unsqueeze(0).to(device)
            mask = mask.unsqueeze(0).to(device)

            with torch.no_grad():
                out = bert_model(
                    input_ids      = ids,
                    attention_mask = mask
                )
                cls_emb = out.last_hidden_state[:, 0, :]
                chunk_embs.append(cls_emb.cpu().float().numpy())

        # mean of all chunks → one embedding per text
        text_emb = np.mean(chunk_embs, axis=0).squeeze()
        batch_embeddings.append(text_emb)

    return np.array(batch_embeddings)   # (batch_size, 768)


# ────────────────────────────────────────────────────────────────
# Test on 3 notes first
# ────────────────────────────────────────────────────────────────
cardiac_df = pd.read_csv('cardiac_discharge_clean.csv', dtype=str)
print(f"Loaded: {len(cardiac_df):,} notes\n")

print("Testing batch embedding on 3 notes...")
test_texts = cardiac_df['text_clean'].iloc[:3].tolist()
test_embs  = embed_batch(test_texts)

print(f"Output shape : {test_embs.shape}")   # (3, 768)
print(f"Mean values  : {test_embs.mean(axis=1)}")
print(f"No NaN       : {not np.isnan(test_embs).any()}")
print("\nTest passed! now run full.\n")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11073.08it/s]
[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
Loaded: 131,390 notes

Testing batch embedding on 3 notes...
Output shape : (3, 768)
Mean values  : [-0.0080734  -0.00865724 -0.00808505]
No NaN       : True

Test passed! now run full.



In [11]:
# ────────────────────────────────────────────────────────────────
# Full batch embedding — GPU optimized + incremental save
# ────────────────────────────────────────────────────────────────
BATCH_SIZE   = 16        # 16 safe in GPU for 1080 Ti
SAVE_EVERY   = 5000      # save on disk after each 5000 note
OUTPUT_DIR   = 'embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

texts       = cardiac_df['text_clean'].tolist()
subject_ids = cardiac_df['subject_id'].tolist()
note_ids    = cardiac_df['note_id'].tolist()

total       = len(texts)
chunk_embs  = []
chunk_meta  = []
saved_count = 0
file_idx    = 0

print(f"Total notes  : {total:,}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Total batches: {total // BATCH_SIZE + 1:,}")
print(f"Save every   : {SAVE_EVERY} notes\n")

for batch_start in range(0, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)

    batch_texts = texts[batch_start:batch_end]
    batch_sids  = subject_ids[batch_start:batch_end]
    batch_nids  = note_ids[batch_start:batch_end]

    try:
        embs = embed_batch(batch_texts)   # (batch, 768)
    except Exception as e:
        print(f"Error at batch {batch_start}: {e}")
        embs = np.zeros((len(batch_texts), 768))

    chunk_embs.append(embs)
    for i, (sid, nid) in enumerate(zip(batch_sids, batch_nids)):
        chunk_meta.append({
            'note_id'   : nid,
            'subject_id': sid,
            'global_idx': batch_start + i
        })

    saved_count += len(batch_texts)

    # Progress
    if (batch_start // BATCH_SIZE + 1) % 50 == 0:
        pct = saved_count / total * 100
        print(f"  {saved_count:,}/{total:,} ({pct:.1f}%) done")

    # Incremental save — RAM clear 
    if saved_count % SAVE_EVERY < BATCH_SIZE:
        save_embs = np.vstack(chunk_embs)
        save_path = f'{OUTPUT_DIR}/emb_chunk_{file_idx:04d}.npy'
        np.save(save_path, save_embs)
        print(f"  Saved chunk {file_idx} → {save_embs.shape} "
              f"at {save_path}")

        chunk_embs = []    # RAM free 
        file_idx  += 1
        gc.collect()
        torch.cuda.empty_cache()

# Last remaining batch save
if chunk_embs:
    save_embs = np.vstack(chunk_embs)
    save_path = f'{OUTPUT_DIR}/emb_chunk_{file_idx:04d}.npy'
    np.save(save_path, save_embs)
    print(f"  Final chunk saved → {save_embs.shape}")

# Meta save
meta_df = pd.DataFrame(chunk_meta)
meta_df.to_csv(f'{OUTPUT_DIR}/embedding_index.csv', index=False)

print(f"\nAll done!")
print(f"Total embedded : {saved_count:,}")
print(f"Chunks saved   : {file_idx + 1}")
print(f"Index saved    : {OUTPUT_DIR}/embedding_index.csv")

Total notes  : 131,390
Batch size   : 16
Total batches: 8,212
Save every   : 5000 notes

  800/131,390 (0.6%) done
  1,600/131,390 (1.2%) done
  2,400/131,390 (1.8%) done
  3,200/131,390 (2.4%) done
  4,000/131,390 (3.0%) done
  4,800/131,390 (3.7%) done
  Saved chunk 0 → (5008, 768) at embeddings/emb_chunk_0000.npy
  5,600/131,390 (4.3%) done
  6,400/131,390 (4.9%) done
  7,200/131,390 (5.5%) done
  8,000/131,390 (6.1%) done
  8,800/131,390 (6.7%) done
  9,600/131,390 (7.3%) done
  Saved chunk 1 → (4992, 768) at embeddings/emb_chunk_0001.npy
  10,400/131,390 (7.9%) done
  11,200/131,390 (8.5%) done
  12,000/131,390 (9.1%) done
  12,800/131,390 (9.7%) done
  13,600/131,390 (10.4%) done
  14,400/131,390 (11.0%) done
  Saved chunk 2 → (5008, 768) at embeddings/emb_chunk_0002.npy
  15,200/131,390 (11.6%) done
  16,000/131,390 (12.2%) done
  16,800/131,390 (12.8%) done
  17,600/131,390 (13.4%) done
  18,400/131,390 (14.0%) done
  19,200/131,390 (14.6%) done
  20,000/131,390 (15.2%) done
  

In [12]:
import glob
import numpy as np
import pandas as pd

# Merge all chunks
chunk_files = sorted(glob.glob('embeddings/emb_chunk_*.npy'))
print(f"Found {len(chunk_files)} chunk files")

all_embs = np.vstack([np.load(f) for f in chunk_files])
np.save('clinical_embeddings.npy', all_embs)

meta_df = pd.read_csv('embeddings/embedding_index.csv')

print(f"Final shape : {all_embs.shape}")
print(f"Meta shape  : {meta_df.shape}")
print(f"Size        : {all_embs.nbytes/1e6:.1f} MB")
print(f"NaN check   : {np.isnan(all_embs).sum()} NaNs")
print(f"Match check : {'OK' if all_embs.shape[0] == len(meta_df) else ' Mismatch'}")

Found 27 chunk files
Final shape : (131390, 768)
Meta shape  : (131390, 3)
Size        : 403.6 MB
NaN check   : 0 NaNs
Match check : OK


In [ ]:
# Quick sanity check — embeddings distribution
print("Embedding stats:")
print(f"  Mean : {all_embs.mean():.4f}")
print(f"  Std  : {all_embs.std():.4f}")
print(f"  Min  : {all_embs.min():.4f}")
print(f"  Max  : {all_embs.max():.4f}")

# Sample 5 embeddings's norm check
norms = np.linalg.norm(all_embs[:5], axis=1)
print(f"\nFirst 5 embedding norms: {norms.round(4)}")
print("(Should be non-zero and similar range)")

Embedding stats:
  Mean : -0.0080
  Std  : 0.5254
  Min  : -11.0214
  Max  : 1.9210

First 5 embedding norms: [14.746  15.5232 14.7756 14.693  14.5099]
(Should be non-zero and similar range)


In [ ]:
import os
print("Current dir:", os.getcwd())

Current dir: d:\heart_disease\ecg_mulmodal_project


### ClinicalBERT Fine-tuning for Arrhythmia

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoTokenizer, AutoModel,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda')
print(f"Device: {device}")

# ── Load matched data
# cardiac notes + subject_id + labels
df_notes  = pd.read_csv('cardiac_discharge_clean.csv',
                         dtype=str)
df_labels = pd.read_csv('final_dataset_matched.csv',
                         dtype=str)

print(f"Notes  : {len(df_notes):,}")
print(f"Labeled: {len(df_labels):,}")

# subject_id match and labels add 
df_notes['subject_id'] = df_notes['subject_id'].astype(str)
df_labels['subject_id'] = df_labels['subject_id'].astype(str)

# Labels load
labels_arr    = np.load('labels_final.npy')
idx_train_arr = np.load('idx_train.npy')
idx_val_arr   = np.load('idx_val.npy')
idx_test_arr  = np.load('idx_test.npy')

# Embedding index — 
emb_index = pd.read_csv('embeddings/embedding_index.csv',
                          dtype=str)
emb_index['subject_id'] = emb_index['subject_id'].astype(str)

print(f"Emb index: {len(emb_index):,}")
print("Data loaded!")

d:\ecg_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Notes  : 131,390
Labeled: 16,203
Emb index: 131,390
Data loaded!


In [4]:
# ── Build subject_id → label mapping
# df_labels mein subject_id order = labels_final.npy order

df_labels_reset = df_labels.reset_index(drop=True)
sid_to_label    = {}

for i, row in df_labels_reset.iterrows():
    sid_to_label[str(row['subject_id'])] = int(labels_arr[i])

print(f"Labeled patients: {len(sid_to_label):,}")
print(f"Label distribution: {pd.Series(list(sid_to_label.values())).value_counts().to_dict()}")


df_notes['label'] = df_notes['subject_id'].map(sid_to_label)
df_notes = df_notes.dropna(subset=['label'])
df_notes['label'] = df_notes['label'].astype(int)

print(f"\nNotes with labels: {len(df_notes):,}")
print(f"Label dist in notes:")
print(df_notes['label'].value_counts())

Labeled patients: 16,203
Label distribution: {0: 8640, 1: 4871, 2: 2692}

Notes with labels: 34,005
Label dist in notes:
label
0    17305
1     9592
2     7108
Name: count, dtype: int64


In [5]:
# ── Train/val/test split 
train_sids = set(
    df_labels_reset.iloc[idx_train_arr]['subject_id'].astype(str)
)
val_sids   = set(
    df_labels_reset.iloc[idx_val_arr]['subject_id'].astype(str)
)
test_sids  = set(
    df_labels_reset.iloc[idx_test_arr]['subject_id'].astype(str)
)

df_train = df_notes[df_notes['subject_id'].isin(train_sids)].copy()
df_val   = df_notes[df_notes['subject_id'].isin(val_sids)].copy()
df_test  = df_notes[df_notes['subject_id'].isin(test_sids)].copy()

print(f"Train notes: {len(df_train):,}")
print(f"Val notes  : {len(df_val):,}")
print(f"Test notes : {len(df_test):,}")

df_train = df_train.groupby('subject_id').last().reset_index()
df_val   = df_val.groupby('subject_id').last().reset_index()
df_test  = df_test.groupby('subject_id').last().reset_index()

print(f"\nAfter per-patient dedup:")
print(f"Train patients: {len(df_train):,}")
print(f"Val patients  : {len(df_val):,}")
print(f"Test patients : {len(df_test):,}")

# Text column check
text_col = 'text_clean' if 'text_clean' in df_train.columns else 'text'
print(f"\nUsing text column: '{text_col}'")
print(f"Sample text (100 chars): {df_train[text_col].iloc[0][:100]}")

Train notes: 23,764
Val notes  : 3,324
Test notes : 6,917

After per-patient dedup:
Train patients: 11,341
Val patients  : 1,621
Test patients : 3,241

Using text column: 'text_clean'
Sample text (100 chars): allergies: sulfa (sulfonamide antibiotics) / codeine / bactrim chief complaint: shortness of breath 


In [6]:
# ── Dataset class (FIXED)
class CardiacTextDataset(Dataset):
    def __init__(self, df, tokenizer, text_col,
                 max_len=512, stride=256, max_chunks=6):  # ← Change to 6
        self.texts     = df[text_col].fillna('').tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.stride    = stride
        self.max_chunks = max_chunks   # 6 chunks (same as original)

    def get_chunks(self, text):
        enc = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=False,
            padding=False,
            add_special_tokens=True
        )
        ids  = enc['input_ids'][0]
        mask = enc['attention_mask'][0]
        total = len(ids)

        chunks_ids, chunks_mask = [], []
        start = 0
        count = 0
        while start < total and count < self.max_chunks:
            end = min(start + self.max_len, total)
            c_ids = ids[start:end]
            c_mask = mask[start:end]

            # Pad to max_len
            pad = self.max_len - len(c_ids)
            if pad > 0:
                c_ids = torch.cat([c_ids,
                    torch.zeros(pad, dtype=torch.long)])
                c_mask = torch.cat([c_mask,
                    torch.zeros(pad, dtype=torch.long)])

            chunks_ids.append(c_ids)
            chunks_mask.append(c_mask)
            start += self.stride
            count += 1

        # Stack: (n_chunks, max_len)
        return (torch.stack(chunks_ids),
                torch.stack(chunks_mask))

    def __getitem__(self, idx):
        ids, mask = self.get_chunks(self.texts[idx])
        label = self.labels[idx]
        return ids, mask, torch.tensor(label, dtype=torch.long)

In [7]:
# ── Fine-tune model class
class FineTunedClinicalBERT(nn.Module):
    def __init__(self, bert_model, num_classes=3,
                 hidden_dim=256, dropout=0.3):
        super().__init__()
        self.bert = bert_model

        # Classifier on top of CLS pooling
        self.classifier = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout * 0.7),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        """
        input_ids    : (batch, n_chunks, max_len)
        attention_mask: (batch, n_chunks, max_len)
        """
        B, C, L = input_ids.shape

        # Reshape — process all chunks at once
        ids_flat  = input_ids.view(B * C, L)
        mask_flat = attention_mask.view(B * C, L)

        out = self.bert(
            input_ids      = ids_flat,
            attention_mask = mask_flat
        )
        cls_embs = out.last_hidden_state[:, 0, :]  # (B*C, 768)

        # Reshape + mean pool across chunks
        cls_embs = cls_embs.view(B, C, 768)
        pooled   = cls_embs.mean(dim=1)             # (B, 768)

        return self.classifier(pooled)

    def get_embedding(self, input_ids, attention_mask):
        """For new embedding generation"""
        B, C, L = input_ids.shape
        ids_flat  = input_ids.view(B * C, L)
        mask_flat = attention_mask.view(B * C, L)

        with torch.no_grad():
            out = self.bert(
                input_ids      = ids_flat,
                attention_mask = mask_flat
            )
            cls_embs = out.last_hidden_state[:, 0, :]
            cls_embs = cls_embs.view(B, C, 768)
            pooled   = cls_embs.mean(dim=1)
        return pooled

In [8]:
# ── Dataset class (FIXED with __len__)
class CardiacTextDataset(Dataset):
    def __init__(self, df, tokenizer, text_col,
                 max_len=512, stride=256, max_chunks=6):
        self.texts     = df[text_col].fillna('').tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.stride    = stride
        self.max_chunks = max_chunks

    def __len__(self):
        return len(self.texts)   # ← ADD THIS LINE

    def get_chunks(self, text):
        enc = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=False,
            padding=False,
            add_special_tokens=True
        )
        ids  = enc['input_ids'][0]
        mask = enc['attention_mask'][0]
        total = len(ids)

        chunks_ids, chunks_mask = [], []
        start = 0
        count = 0
        while start < total and count < self.max_chunks:
            end = min(start + self.max_len, total)
            c_ids = ids[start:end]
            c_mask = mask[start:end]

            # Pad to max_len
            pad = self.max_len - len(c_ids)
            if pad > 0:
                c_ids = torch.cat([c_ids, torch.zeros(pad, dtype=torch.long)])
                c_mask = torch.cat([c_mask, torch.zeros(pad, dtype=torch.long)])

            chunks_ids.append(c_ids)
            chunks_mask.append(c_mask)
            start += self.stride
            count += 1

        return (torch.stack(chunks_ids), torch.stack(chunks_mask))

    def __getitem__(self, idx):
        ids, mask = self.get_chunks(self.texts[idx])
        label = self.labels[idx]
        return ids, mask, torch.tensor(label, dtype=torch.long)

In [9]:
# ── Setup training
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_base  = AutoModel.from_pretrained(MODEL_NAME)

ft_model = FineTunedClinicalBERT(
    bert_model  = bert_base,
    num_classes = 3,
    hidden_dim  = 256,
    dropout     = 0.3
).to(device)

total_params = sum(p.numel() for p in ft_model.parameters())
trainable    = sum(p.numel() for p in ft_model.parameters()
                   if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")

# Datasets
train_ds = CardiacTextDataset(
    df_train, tokenizer, text_col,
    max_len=512, stride=256, max_chunks=4
)
val_ds   = CardiacTextDataset(
    df_val, tokenizer, text_col,
    max_len=512, stride=256, max_chunks=4
)
test_ds  = CardiacTextDataset(
    df_test, tokenizer, text_col,
    max_len=512, stride=256, max_chunks=4
)

# Loaders — batch_size=4 (BERT is heavy )
train_ldr = DataLoader(
    train_ds, batch_size=4,
    shuffle=True, num_workers=0,
    pin_memory=True
)
val_ldr   = DataLoader(
    val_ds, batch_size=4,
    shuffle=False, num_workers=0
)
test_ldr  = DataLoader(
    test_ds, batch_size=4,
    shuffle=False, num_workers=0
)

print(f"\nTrain batches: {len(train_ldr):,}")
print(f"Val batches  : {len(val_ldr):,}")
print(f"Test batches : {len(test_ldr):,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10490.37it/s]
[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params    : 108,540,931
Trainable params: 108,540,931

Train batches: 2,836
Val batches  : 406
Test batches : 811


In [10]:
# ── Optimizer — BERT ke liye differential LR
# BERT layers: small LR
# Classifier: large LR

bert_params       = list(ft_model.bert.parameters())
classifier_params = list(ft_model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {'params': bert_params,
     'lr': 2e-5, 'weight_decay': 0.01},
    {'params': classifier_params,
     'lr': 2e-4, 'weight_decay': 0.01}
])

# Warmup scheduler
EPOCHS          = 8
total_steps     = len(train_ldr) * EPOCHS
warmup_steps    = total_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

# Class weights
class_weights = np.load('class_weights.npy')
cw = torch.tensor(class_weights,
                   dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(
    weight=cw, label_smoothing=0.05
)

print(f"Total steps  : {total_steps:,}")
print(f"Warmup steps : {warmup_steps:,}")
print("Setup complete!")

Total steps  : 22,688
Warmup steps : 2,268
Setup complete!


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# ADD THIS CELL BEFORE CREATING DATALOADERS
# ═══════════════════════════════════════════════════════════════════════════

def collate_fn(batch):
    """
    Custom collate function for variable number of chunks per sample
    
    Args:
        batch: list of tuples (ids, mask, label)
            ids: (n_chunks, 512) - can vary per sample
            mask: (n_chunks, 512)
            label: int
    
    Returns:
        input_ids: (batch_size, max_chunks, 512)
        attention_mask: (batch_size, max_chunks, 512)
        labels: (batch_size,)
    """
    
    # Find max chunks in this batch
    max_chunks = max([item[0].shape[0] for item in batch])
    
    batch_ids = []
    batch_mask = []
    batch_labels = []
    
    for ids, mask, label in batch:
        n_chunks = ids.shape[0]
        pad_chunks = max_chunks - n_chunks
        
        # Pad if needed
        if pad_chunks > 0:
            # Pad with zeros
            pad_ids = torch.zeros(pad_chunks, 512, dtype=torch.long)
            pad_mask = torch.zeros(pad_chunks, 512, dtype=torch.long)
            ids = torch.cat([ids, pad_ids], dim=0)
            mask = torch.cat([mask, pad_mask], dim=0)
        
        batch_ids.append(ids)
        batch_mask.append(mask)
        batch_labels.append(label)
    
    # Stack to tensors
    input_ids = torch.stack(batch_ids)      # (B, max_chunks, 512)
    attention_mask = torch.stack(batch_mask) # (B, max_chunks, 512)
    labels = torch.tensor(batch_labels, dtype=torch.long)
    
    return input_ids, attention_mask, labels

In [12]:
# ── Recreate dataloaders with custom collate

train_ldr = DataLoader(
    train_ds, 
    batch_size=4,           # Smaller batch for BERT
    shuffle=True, 
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn   # ← ADD THIS
)

val_ldr = DataLoader(
    val_ds, 
    batch_size=4,
    shuffle=False, 
    num_workers=0,
    collate_fn=collate_fn   # ← ADD THIS
)

test_ldr = DataLoader(
    test_ds, 
    batch_size=4,
    shuffle=False, 
    num_workers=0,
    collate_fn=collate_fn   # ← ADD THIS
)

print(f"Train batches: {len(train_ldr):,}")
print(f"Val batches:   {len(val_ldr):,}")
print(f"Test batches:  {len(test_ldr):,}")

Train batches: 2,836
Val batches:   406
Test batches:  811


In [13]:
# Check if model is actually on GPU
print(f"Model on CUDA: {next(ft_model.parameters()).is_cuda}")
print(f"Model device: {next(ft_model.parameters()).device}")

# Check data loading speed
import time

start = time.time()
for i, (ids, mask, y) in enumerate(train_ldr):
    if i >= 5:
        break
    ids = ids.to(device, non_blocking=True)
    mask = mask.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)
    
    # Simple forward pass
    with torch.no_grad():
        out = ft_model(ids, mask)
    print(f"Batch {i+1}: ids shape = {ids.shape}")
    
end = time.time()
print(f"\n5 batches took: {end-start:.2f} seconds")
print(f"Estimated per batch: {(end-start)/5:.2f} seconds")
print(f"Estimated per epoch: {(end-start)/5 * len(train_ldr):.0f} seconds (~{(end-start)/5 * len(train_ldr)/60:.1f} minutes)")

Model on CUDA: True
Model device: cuda:0
Batch 1: ids shape = torch.Size([4, 4, 512])
Batch 2: ids shape = torch.Size([4, 4, 512])
Batch 3: ids shape = torch.Size([4, 4, 512])
Batch 4: ids shape = torch.Size([4, 4, 512])
Batch 5: ids shape = torch.Size([4, 4, 512])

5 batches took: 1.79 seconds
Estimated per batch: 0.36 seconds
Estimated per epoch: 1015 seconds (~16.9 minutes)


In [14]:
# ── Training loop
import time

PATIENCE      = 3
best_val_loss = float('inf')
best_weights  = None
patience_ctr  = 0
history       = {'tl':[], 'vl':[], 'ta':[], 'va':[]}

print("Fine-tuning ClinicalBERT...\n")
print(f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | "
      f"{'VlLoss':>8} | {'VlAcc':>7} | {'Time':>6}")
print("-"*55)

for epoch in range(1, EPOCHS+1):
    t0 = time.time()

    # ── Train
    ft_model.train()
    tl, tc, tt = 0, 0, 0

    for ids, mask, y in train_ldr:
        ids  = ids.to(device)
        mask = mask.to(device)
        y    = y.to(device)

        optimizer.zero_grad()
        out  = ft_model(ids, mask)
        loss = criterion(out, y)
        loss.backward()

        # Gradient clipping
        nn.utils.clip_grad_norm_(
            ft_model.parameters(), 1.0
        )
        optimizer.step()
        scheduler.step()

        tl += loss.item() * len(y)
        tc += (out.argmax(1)==y).sum().item()
        tt += len(y)

    # ── Validation
    ft_model.eval()
    vl, vc, vt = 0, 0, 0

    with torch.no_grad():
        for ids, mask, y in val_ldr:
            ids, mask = ids.to(device), mask.to(device)
            y         = y.to(device)
            out  = ft_model(ids, mask)
            loss = criterion(out, y)
            vl += loss.item() * len(y)
            vc += (out.argmax(1)==y).sum().item()
            vt += len(y)

    tr_loss = tl/tt; tr_acc = tc/tt
    vl_loss = vl/vt; vl_acc = vc/vt
    elapsed = time.time() - t0

    history['tl'].append(tr_loss)
    history['vl'].append(vl_loss)
    history['ta'].append(tr_acc)
    history['va'].append(vl_acc)

    print(f"{epoch:>3} | {tr_loss:>8.4f} | "
          f"{tr_acc:>7.4f} | {vl_loss:>8.4f} | "
          f"{vl_acc:>7.4f} | {elapsed:>5.0f}s")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_weights  = {
            k: v.cpu().clone()
            for k, v in ft_model.state_dict().items()
        }
        patience_ctr = 0
        print(f"      ✅ Best model saved!")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

ft_model.load_state_dict(best_weights)
torch.save(best_weights, 'clinicalbert_finetuned.pth')
print("\nSaved: clinicalbert_finetuned.pth ✅")

Fine-tuning ClinicalBERT...

 Ep |   TrLoss |   TrAcc |   VlLoss |   VlAcc |   Time
-------------------------------------------------------
  1 |   1.1186 |  0.4161 |   1.0411 |  0.5460 |  3110s
      ✅ Best model saved!
  2 |   1.0367 |  0.4805 |   0.9902 |  0.3825 |  2953s
      ✅ Best model saved!
  3 |   1.0342 |  0.4870 |   1.0380 |  0.3572 |  2952s
  4 |   1.0057 |  0.5186 |   0.9932 |  0.5626 |  2951s
  5 |   0.9975 |  0.5187 |   0.9770 |  0.5645 |  2952s
      ✅ Best model saved!
  6 |   0.9985 |  0.5198 |   0.9822 |  0.5676 |  2951s
  7 |   0.9889 |  0.5221 |   0.9831 |  0.5700 |  2954s
  8 |   0.9851 |  0.5343 |   0.9777 |  0.5700 |  2949s

Early stopping at epoch 8

Saved: clinicalbert_finetuned.pth ✅


In [15]:
# ── Test evaluation — fine-tuned text-only
ft_model.eval()
all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for ids, mask, y in test_ldr:
        ids, mask = ids.to(device), mask.to(device)
        out   = ft_model(ids, mask)
        probs = torch.softmax(out, 1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_true.extend(y.numpy())

all_probs = np.array(all_probs)
ft_auc    = roc_auc_score(all_true, all_probs,
                            multi_class='ovr',
                            average='macro')
ft_acc    = (np.array(all_preds)==np.array(all_true)).mean()

print("── Fine-tuned ClinicalBERT Results ──")
print(classification_report(
    all_true, all_preds,
    target_names=['NORMAL','MILD','SEVERE'],
    digits=4
))
print(f"AUC-ROC  : {ft_auc:.4f}")
print(f"Accuracy : {ft_acc*100:.2f}%")
print(f"\nComparison:")
print(f"  Before fine-tuning: AUC=0.5937, Acc=44.8%")
print(f"  After fine-tuning : AUC={ft_auc:.4f}, "
      f"Acc={ft_acc*100:.1f}%")

── Fine-tuned ClinicalBERT Results ──
              precision    recall  f1-score   support

      NORMAL     0.6030    0.8027    0.6887      1728
        MILD     0.0000    0.0000    0.0000       974
      SEVERE     0.4431    0.7737    0.5635       539

    accuracy                         0.5566      3241
   macro avg     0.3487    0.5254    0.4174      3241
weighted avg     0.3952    0.5566    0.4609      3241

AUC-ROC  : 0.6605
Accuracy : 55.66%

Comparison:
  Before fine-tuning: AUC=0.5937, Acc=44.8%
  After fine-tuning : AUC=0.6605, Acc=55.7%


In [22]:
# ── Phase 1 model best hai — use this
# AUC 0.6605 >> original 0.5937
# Acc 55.7% >> original 44.8%
# NORMAL class preserved

print("Using Phase 1 model (AUC=0.6605)")
print("Generating embeddings — fast mode...\n")

# Fast Dataset — sirf first 512 tokens
class FastEmbDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = [str(t) for t in texts]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            truncation     = True,
            padding        = 'max_length',
            return_tensors = 'pt'
        )
        ids  = enc['input_ids'][0].unsqueeze(0)
        mask = enc['attention_mask'][0].unsqueeze(0)
        return ids, mask

# All 16203 patients
df_all_pts = pd.concat(
    [df_train, df_val, df_test],
    ignore_index=True
)
print(f"Total patients: {len(df_all_pts):,}")

fast_ds  = FastEmbDataset(
    df_all_pts[text_col].fillna('').tolist(),
    tokenizer, max_len=512
)
fast_ldr = DataLoader(
    fast_ds, batch_size=32,
    shuffle=False, num_workers=0,
    pin_memory=True
)
print(f"Batches: {len(fast_ldr):,}")
print("Generating... (should take ~15-20 min)\n")

# Phase 1 model
ft_model.eval()
all_embs_ft = []

with torch.no_grad():
    for bidx, (ids, mask) in enumerate(fast_ldr):
        ids  = ids.to(device)
        mask = mask.to(device)
        emb  = ft_model.get_embedding(ids, mask)
        all_embs_ft.append(emb.cpu().numpy())

        if (bidx+1) % 50 == 0:
            done = min((bidx+1)*32, len(df_all_pts))
            pct  = done/len(df_all_pts)*100
            print(f"  {done:,}/{len(df_all_pts):,} ({pct:.1f}%)")

all_embs_ft = np.vstack(all_embs_ft)
print(f"\nDone! Shape: {all_embs_ft.shape}")

Using Phase 1 model (AUC=0.6605)
Generating embeddings — fast mode...

Total patients: 16,203
Batches: 507
Generating... (should take ~15-20 min)

  1,600/16,203 (9.9%)
  3,200/16,203 (19.7%)
  4,800/16,203 (29.6%)
  6,400/16,203 (39.5%)
  8,000/16,203 (49.4%)
  9,600/16,203 (59.2%)
  11,200/16,203 (69.1%)
  12,800/16,203 (79.0%)
  14,400/16,203 (88.9%)
  16,000/16,203 (98.7%)

Done! Shape: (16203, 768)


In [23]:
# ── Labels order mein arrange karo
sid_to_emb_ft = {}
for sid, emb in zip(
    df_all_pts['subject_id'].tolist(),
    all_embs_ft
):
    sid_to_emb_ft[str(sid)] = emb

ft_embs_final = np.zeros(
    (len(df_labels_reset), 768), dtype=np.float32
)
found = 0
for i, row in df_labels_reset.iterrows():
    sid = str(row['subject_id'])
    if sid in sid_to_emb_ft:
        ft_embs_final[i] = sid_to_emb_ft[sid]
        found += 1

print(f"Matched : {found:,}/{len(df_labels_reset):,}")
print(f"Missing : {len(df_labels_reset)-found:,}")

np.save('text_embeddings_finetuned.npy', ft_embs_final)
print(f"✅ Saved: text_embeddings_finetuned.npy")
print(f"Shape  : {ft_embs_final.shape}")

Matched : 16,203/16,203
Missing : 0
✅ Saved: text_embeddings_finetuned.npy
Shape  : (16203, 768)


In [24]:
# ── Quick silhouette check
from sklearn.metrics import silhouette_score

# Sample 500 for speed
sample_idx = np.random.choice(
    len(ft_embs_final), 500, replace=False
)
sample_embs   = ft_embs_final[sample_idx]
sample_labels = np.load('labels_final.npy')[sample_idx]

sil_orig = -0.0147   # original ClinicalBERT
sil_ft   = silhouette_score(sample_embs, sample_labels)

print("\n── Embedding Quality ──")
print(f"Original BERT silhouette : {sil_orig:.4f}")
print(f"Fine-tuned  silhouette   : {sil_ft:.4f}")
print(f"Improvement              : {sil_ft-sil_orig:+.4f}")
print(f"\n{'Better' if sil_ft > sil_orig else 'Similar'} "
      f"class discriminability!")


── Embedding Quality ──
Original BERT silhouette : -0.0147
Fine-tuned  silhouette   : -0.0723
Improvement              : -0.0576

Similar class discriminability!


In [26]:
# ── Updated ablation table
print("\n" + "="*60)
print("  FINAL ABLATION TABLE — paper ready")
print("="*60)
print(f"{'Method':<38} {'AUC':>7} {'Acc':>7}")
print("-"*54)
rows = [
    ("Text only — fine-tuned BERT†",   0.6605, 0.5566),
    ("ECG only (1D-ResNet)",            0.9392, 0.8497),
    ("ECG + Clinical intervals",        0.9820, 0.9340),
    ("Full Fusion CGA (original text)", 0.9836, 0.9432),
    ("Full Fusion CGA (ft text) ★",    0.0000, 0.0000),
]
for method, auc, acc in rows:
    print(f"{method:<38} {auc:>7.4f} {acc:>7.4f}")
print("="*60)
print("★ = retrain fusion.ipynb se — TBD")
print("† = limited by label-modality mismatch")


  FINAL ABLATION TABLE — paper ready
Method                                     AUC     Acc
------------------------------------------------------
Text only — fine-tuned BERT†            0.6605  0.5566
ECG only (1D-ResNet)                    0.9392  0.8497
ECG + Clinical intervals                0.9820  0.9340
Full Fusion CGA (original text)         0.9836  0.9432
Full Fusion CGA (ft text) ★             0.0000  0.0000
★ = retrain fusion.ipynb se — TBD
† = limited by label-modality mismatch
